# Bronze source ingestion

Validates and loads the 28 manifest-listed PowerSchool, Canvas, Moodle, and NWEA source files into pre-created source-qualified Delta tables. Source values remain strings and replay-safe lineage metadata is added; business typing and conformance belong in Silver.

## Deployment

Run `EDU_BR_CreateSchema` before this notebook. The optional installer creates or reuses the Bronze Lakehouse, uploads the validated source package, deploys both notebooks, and orchestrates them in dependency order.

For manual development, attach the Bronze Lakehouse as the default and upload `sample-data/manifest.json` plus the four source folders beneath `Files/landing/education-analytics`.

In [ ]:
# Deployment parameters
source_root = "Files/landing/education-analytics"
manifest_path = f"{source_root}/manifest.json"
source_instance_id = "synthetic-district-v2"
observation_cutoff = None  # ISO-8601 timestamp, or None when the extract has no declared cutoff
skip_unchanged_files = True
fail_on_validation_error = True
exit_notebook_on_success = True

import csv
import io
import json
import re
import uuid
from datetime import datetime, timezone

from pyspark.sql import DataFrame, Window, functions as F, types as T

load_batch_id = str(uuid.uuid4())
load_started_at = datetime.now(timezone.utc)
ingested_at_utc = load_started_at.replace(tzinfo=None)
allowed_sources = {"PowerSchool", "Canvas", "Moodle", "NWEA"}
print(f"Load batch: {load_batch_id}")

## Ingestion helpers

The helpers use Fabric `notebookutils` for OneLake discovery and Spark's binary-file reader for SHA-256 validation. All contract checks complete before any Bronze rows are written. Physical table creation and drift validation belong to `EDU_BR_CreateSchema`.

In [ ]:
def read_json(path: str) -> dict:
    return json.loads(notebookutils.fs.head(path, 1024 * 1024))


def list_csv_files(root: str) -> dict[str, str]:
    discovered = {}

    def visit(path: str, relative_prefix: str) -> None:
        for item in notebookutils.fs.ls(path):
            relative_path = f"{relative_prefix}/{item.name}" if relative_prefix else item.name
            if item.isDir:
                visit(item.path, relative_path.rstrip("/"))
            elif item.name.lower().endswith(".csv"):
                discovered[relative_path] = item.path

    for source in sorted(allowed_sources):
        visit(f"{root}/{source}", source)
    return discovered


def read_csv_header(path: str) -> list[str]:
    first_line = notebookutils.fs.head(path, 1024 * 1024).splitlines()[0]
    return next(csv.reader(io.StringIO(first_line)))


def file_sha256(path: str) -> str:
    row = (spark.read.format("binaryFile").load(path)
           .select(F.sha2(F.col("content"), 256).alias("sha256"))
           .first())
    if row is None:
        raise ValueError(f"Could not read bytes from {path}")
    return row.sha256.lower()


def source_schema(columns: list[str]) -> T.StructType:
    return T.StructType(
        [T.StructField(column, T.StringType(), True) for column in columns]
        + [T.StructField("_corrupt_record", T.StringType(), True)]
    )


def read_source_csv(path: str, columns: list[str]) -> DataFrame:
    return (spark.read
            .option("header", True)
            .option("encoding", "UTF-8")
            .option("sep", ",")
            .option("quote", '"')
            .option("escape", '"')
            .option("multiLine", True)
            .option("nullValue", "\\N")
            .option("emptyValue", "")
            .option("mode", "PERMISSIVE")
            .option("columnNameOfCorruptRecord", "_corrupt_record")
            .schema(source_schema(columns))
            .csv(path))


def normalized_table_name(source: str, relative_path: str) -> str:
    stem = relative_path.rsplit("/", 1)[-1].rsplit(".", 1)[0]
    return re.sub(r"[^a-z0-9]+", "_", f"{source}_{stem}".lower()).strip("_")

## Validate the landing set and identify changes

The notebook always checks the exact file set, headers, and SHA-256 hashes. When `skip_unchanged_files` is enabled, a file whose hash matches its latest successful zero-rejection audit record and whose target table still exists bypasses CSV parsing, row counting, and Delta rewriting. New, changed, or incomplete tables follow the full validation path before any writes begin.

In [ ]:
manifest = read_json(manifest_path)
tables = manifest.get("tables", [])
manifest_sources = {table["source"] for table in tables}

contract_errors = []
if len(tables) != 28:
    contract_errors.append(f"Manifest declares {len(tables)} tables; expected 28")
if manifest_sources != allowed_sources:
    contract_errors.append(
        f"Manifest sources are {sorted(manifest_sources)}; expected {sorted(allowed_sources)}"
    )

expected_paths = {table["path"] for table in tables}
if len(expected_paths) != len(tables):
    contract_errors.append("Manifest contains duplicate table paths")

required_tables = {
    normalized_table_name(table["source"], table["path"])
    for table in tables
} | {"ingestion_audit", "ingestion_quarantine"}
missing_tables = sorted(table for table in required_tables if not spark.catalog.tableExists(table))
if missing_tables:
    contract_errors.append(
        "Missing Bronze tables: " + ", ".join(missing_tables)
        + "; run EDU_BR_CreateSchema before ingestion"
    )

discovered_files = list_csv_files(source_root)
actual_paths = set(discovered_files)
missing_paths = sorted(expected_paths - actual_paths)
extra_paths = sorted(actual_paths - expected_paths)
if missing_paths:
    contract_errors.append(f"Missing CSV files: {missing_paths}")
if extra_paths:
    contract_errors.append(f"Unexpected CSV files: {extra_paths}")

latest_audit_by_path = {}
if skip_unchanged_files and spark.catalog.tableExists("ingestion_audit"):
    latest_window = Window.partitionBy("SourceFileName").orderBy(F.col("FinishedAtUtc").desc())
    latest_audit_rows = (spark.table("ingestion_audit")
        .filter(
            (F.col("SourceInstanceId") == source_instance_id)
            & F.col("Status").isin("SUCCEEDED", "SKIPPED_UNCHANGED")
        )
        .withColumn("AuditRank", F.row_number().over(latest_window))
        .filter(F.col("AuditRank") == 1)
        .drop("AuditRank")
        .collect())
    latest_audit_by_path = {row.SourceFileName: row.asDict() for row in latest_audit_rows}

validated_tables = {}
contract_results = []
for table in tables:
    relative_path = table["path"]
    if relative_path not in discovered_files:
        continue

    full_path = discovered_files[relative_path]
    target_table = normalized_table_name(table["source"], relative_path)
    expected_columns = table["columns"]
    actual_columns = read_csv_header(full_path)
    actual_hash = file_sha256(full_path)
    expected_rows = int(table["rows"])
    previous_audit = latest_audit_by_path.get(relative_path)
    table_errors = []

    if actual_columns != expected_columns:
        table_errors.append(f"header expected {expected_columns}, received {actual_columns}")
    if actual_hash != table["sha256"].lower():
        table_errors.append(f"sha256 expected {table['sha256']}, received {actual_hash}")

    unchanged = (
        not table_errors
        and previous_audit is not None
        and previous_audit["ActualSha256"] == actual_hash
        and int(previous_audit["ActualRows"]) == expected_rows
        and int(previous_audit["RejectedRows"]) == 0
        and spark.catalog.tableExists(target_table)
    )

    frame = None
    actual_rows = expected_rows
    if not unchanged and not table_errors:
        frame = read_source_csv(full_path, expected_columns).cache()
        actual_rows = frame.count()
        if actual_rows != expected_rows:
            table_errors.append(f"rows expected {expected_rows}, received {actual_rows}")

    status = "UNCHANGED" if unchanged else ("VALID" if not table_errors else "INVALID")
    contract_results.append({
        "Source": table["source"],
        "Entity": table["entity"],
        "Path": relative_path,
        "ExpectedRows": expected_rows,
        "ActualRows": actual_rows,
        "Status": status,
        "Errors": "; ".join(table_errors),
    })
    if table_errors:
        contract_errors.append(f"{relative_path}: {'; '.join(table_errors)}")
        if frame is not None:
            frame.unpersist()
    else:
        validated_tables[relative_path] = {
            "frame": frame,
            "actual_hash": actual_hash,
            "actual_rows": actual_rows,
            "unchanged": unchanged,
        }

display(spark.createDataFrame(contract_results).orderBy("Source", "Path"))

if contract_errors:
    for state in validated_tables.values():
        if state["frame"] is not None:
            state["frame"].unpersist()
    raise ValueError("Landing contract validation failed:\n- " + "\n- ".join(contract_errors))

unchanged_count = sum(1 for state in validated_tables.values() if state["unchanged"])
print(f"Validated {len(validated_tables)} files: {unchanged_count} unchanged, {len(validated_tables) - unchanged_count} to write")

## Write changed Delta tables

Each changed CSV becomes one source-qualified Delta table in the default Bronze Lakehouse, such as `powerschool_students` or `canvas_submissions`. Unchanged tables are recorded as `SKIPPED_UNCHANGED` without transforming or rewriting their rows. Changed tables retain replay-safe source-instance replacement, malformed rows go to `ingestion_quarantine`, and every decision goes to `ingestion_audit`.

In [ ]:
audit_schema = spark.table("ingestion_audit").schema

audit_rows = []
load_errors = []
instance_predicate = "SourceInstanceId = '{}'".format(source_instance_id.replace("'", "''"))

for table in tables:
    started_at = datetime.now(timezone.utc).replace(tzinfo=None)
    relative_path = table["path"]
    source_columns = table["columns"]
    target_table = normalized_table_name(table["source"], relative_path)
    state = validated_tables[relative_path]
    frame = state["frame"]
    actual_hash = state["actual_hash"]
    actual_rows = int(state["actual_rows"])
    accepted_rows = actual_rows
    rejected_rows = 0
    status = "SKIPPED_UNCHANGED" if state["unchanged"] else "SUCCEEDED"
    error_message = None

    try:
        if not state["unchanged"]:
            validation_errors = F.when(
                F.col("_corrupt_record").isNotNull(),
                F.array(F.lit("MALFORMED_CSV_RECORD")),
            ).otherwise(F.array().cast(T.ArrayType(T.StringType())))

            enriched = (frame
                .withColumn("ValidationErrors", validation_errors)
                .withColumn("LoadBatchId", F.lit(load_batch_id))
                .withColumn("SourceSystemName", F.lit(table["source"]))
                .withColumn("SourceEntityName", F.lit(table["entity"]))
                .withColumn("SourceInstanceId", F.lit(source_instance_id))
                .withColumn("SourceFileName", F.lit(relative_path))
                .withColumn("IngestedAtUtc", F.lit(ingested_at_utc).cast("timestamp"))
                .withColumn("ObservationCutoff", F.lit(observation_cutoff).cast("timestamp"))
                .withColumn("RawRecordHash", F.sha2(
                    F.to_json(
                        F.struct(*[F.col(column) for column in source_columns]),
                        options={"ignoreNullFields": "false"},
                    ),
                    256,
                )))

            rejected = enriched.filter(F.size("ValidationErrors") > 0)
            accepted = (enriched.filter(F.size("ValidationErrors") == 0)
                        .drop("_corrupt_record", "ValidationErrors"))
            rejected_rows = rejected.count()
            accepted_rows = actual_rows - rejected_rows

            if rejected_rows:
                quarantine = rejected.select(
                    "LoadBatchId", "SourceSystemName", "SourceEntityName",
                    "SourceInstanceId", "SourceFileName", "IngestedAtUtc",
                    "RawRecordHash", "ValidationErrors",
                    F.to_json(F.struct(*[F.col(column) for column in source_columns]),
                              options={"ignoreNullFields": "false"}).alias("RawPayload"),
                    F.col("_corrupt_record").alias("CorruptRecord"),
                )
                (quarantine.write.format("delta").mode("append")
                 .saveAsTable("ingestion_quarantine"))

            (accepted.write.format("delta").mode("overwrite")
             .option("replaceWhere", instance_predicate)
             .saveAsTable(target_table))
    except Exception as exc:
        status = "FAILED"
        error_message = str(exc)[:4000]
        load_errors.append(f"{relative_path}: {error_message}")
    finally:
        if frame is not None:
            frame.unpersist()
        audit_rows.append((
            load_batch_id, table["source"], table["entity"], source_instance_id,
            relative_path, target_table, int(table["rows"]), actual_rows,
            accepted_rows, rejected_rows, table["sha256"], actual_hash,
            status, started_at, datetime.now(timezone.utc).replace(tzinfo=None), error_message,
        ))

audit_frame = spark.createDataFrame(audit_rows, audit_schema)
(audit_frame.write.format("delta").mode("append")
 .saveAsTable("ingestion_audit"))
display(audit_frame.orderBy("SourceSystemName", "SourceFileName"))

if load_errors and fail_on_validation_error:
    raise RuntimeError("Bronze ingestion failed:\n- " + "\n- ".join(load_errors))

## Post-load assertions and pipeline result

These checks count both written and unchanged-skipped tables as successfully processed, while reporting them separately for pipeline monitoring. Attendance rates, missing/late work, identity resolution, and other business metrics remain Silver responsibilities.

In [ ]:
manifest_counts = {table["path"]: int(table["rows"]) for table in tables}
expected_table_count = len(tables)
expected_source_rows = sum(manifest_counts.values())

completed_rows = audit_frame.filter(F.col("Status").isin("SUCCEEDED", "SKIPPED_UNCHANGED"))
summary = completed_rows.agg(
    F.count("*").alias("TablesProcessed"),
    F.sum(F.when(F.col("Status") == "SUCCEEDED", 1).otherwise(0)).alias("TablesWritten"),
    F.sum(F.when(F.col("Status") == "SKIPPED_UNCHANGED", 1).otherwise(0)).alias("TablesSkipped"),
    F.sum("ActualRows").alias("SourceRows"),
    F.sum("AcceptedRows").alias("AcceptedRows"),
    F.sum("RejectedRows").alias("RejectedRows"),
).first()

assert summary.TablesProcessed == expected_table_count, f"Processed {summary.TablesProcessed} of {expected_table_count} tables"
assert summary.SourceRows == expected_source_rows, f"Validated {summary.SourceRows} of {expected_source_rows} source rows"

result = {
    "status": "SUCCEEDED",
    "loadBatchId": load_batch_id,
    "sourceInstanceId": source_instance_id,
    "tablesProcessed": summary.TablesProcessed,
    "tablesWritten": summary.TablesWritten,
    "tablesSkipped": summary.TablesSkipped,
    "sourceRows": summary.SourceRows,
    "acceptedRows": summary.AcceptedRows,
    "rejectedRows": summary.RejectedRows,
}
display(spark.createDataFrame([result]))
result_json = json.dumps(result, separators=(",", ":"))
print(result_json)

if exit_notebook_on_success:
    notebookutils.notebook.exit(result_json)